# ColdSite-DTI — the cold-pair volume-matched control (Colab, T4)

**The question.** `cold_pair` requires both the drug and the target to be unseen, so it
throws away every pair where only one of them is. It trains on **15,190 rows**, against
**21,039** for `random`. Part of its accuracy drop could therefore be *less data*
rather than *a harder task*. A reviewer will ask which.

**The control.** Train ColdSite-DTI on the `random` split with its training set cut to
cold-pair's size, 3 seeds. Validation and test are **not** cut, so these three cells are
scored on exactly the same test set as the grid's full `random` cells.

| compare | tells you |
|---|---|
| full `random` vs this control | how much AUROC the lost rows alone cost |
| this control vs `cold_pair` | how much is genuine cold-pair difficulty |

Binary task, DAVIS, matching the 36-run audit grid on Kaggle, which supplies the full
`random` and `cold_pair` cells to compare against.

## Before you run

**Runtime -> Change runtime type -> T4 GPU.** Then run the cells in order. Section 2
asks you to authorise Google Drive -- that click is yours.

## Safe to interrupt

Each finished cell goes to Drive straight away. If Colab disconnects, run the notebook
again from the top: finished seeds are skipped. A seed cut off mid-training starts again.
Expect **~2-4 hours** for all three.

## Why the outputs are renamed

`train.py` names a cut-down run exactly like the full one -- the same
`davis_random_binary_seed1_results.json`. Mixed into the grid's folder, a control cell
would silently replace a real one. So every output here is renamed with a
`_trainsub<N>` suffix and kept in its own Drive folder. **Never copy these into the grid's
results.**


## 1. Check the runtime


In [ ]:
import time, torch

assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU.'
p = torch.cuda.get_device_properties(0)
print(f'GPU   : {p.name}, {p.total_memory/1e9:.1f} GB')
print(f'torch : {torch.__version__}')
BATCH = 64 if p.total_memory / 1e9 >= 14 else 16    # 64 needs 8.7 GB
print(f'batch : {BATCH}')


## 2. Mount Drive

Results live in their own folder, `coldsite-volume-control`, never in a grid folder.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS = '/content/drive/MyDrive/coldsite-volume-control'
os.makedirs(RESULTS, exist_ok=True)
print('results ->', RESULTS)
print('already there:', sorted(f for f in os.listdir(RESULTS) if f.endswith('_results.json')))


## 3. Clone the repo


In [ ]:
REPO = 'https://github.com/Mahim56207/ColdSite-DTI_New.git'
SRC = '/content/ColdSite-DTI_New'
if not os.path.exists(SRC):
    !git clone --branch main {REPO} {SRC}
os.chdir(SRC)
!git pull origin main
!pip install -q tabulate subword-nmt

import importlib, src.model.dataset as _ds
importlib.reload(_ds)
assert hasattr(_ds, 'BINARY_THRESHOLD'), (
    'This checkout predates the binary-label fix -- ColdSite-DTI would crash. '
    'Re-run this cell.')
!git log --oneline -1


## 4. Data and splits


In [ ]:
BASE = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data'
for ds in ('davis', 'kiba'):
    os.makedirs(f'src/data/baselines/deepdta/data/{ds}', exist_ok=True)
    for fname in ('ligands_can.txt', 'proteins.txt', 'Y'):
        target = f'src/data/baselines/deepdta/data/{ds}/{fname}'
        if not os.path.exists(target):
            !curl -sL {BASE}/{ds}/{fname} -o {target}
!python -m src.data.build_splits 2>&1 | grep -E 'davis|leakage'

import pandas as pd

def rows(split, part):
    return len(pd.read_csv(f'data/splits/davis/{split}/{part}.csv'))

FULL = rows('random', 'train')
N_SUB = rows('cold_pair', 'train')        # the volume to match, read from the data
assert (FULL, rows('random', 'valid'), rows('random', 'test')) == (21039, 3006, 6011), \
    'random split differs from the rest of the project'
assert N_SUB == 15190, 'cold_pair split differs from the rest of the project'

print(f'random trains on {FULL:,} rows; cold_pair on {N_SUB:,} ({100*N_SUB/FULL:.1f}%).')
print(f'The control trains random on {N_SUB:,}, and tests on the full {rows("random", "test"):,}.')


## 5. Train the three seeds

Each seed draws a different 15,190-row subset (the subset follows the training seed),
so the spread across seeds includes the luck of the draw, not just initialisation.


In [ ]:
import json, shutil, subprocess, sys

TMP = '/content/vc_tmp'                       # train here, move to Drive when done
SEEDS = [1, 2, 3]
SUFFIX = f'_trainsub{N_SUB}'


def final_name(tag, kind):
    base = {'results': f'{tag}{SUFFIX}_results.json',
            'checkpoint': f'coldsite_dti_{tag}{SUFFIX}.pt',
            'history': f'coldsite_dti_{tag}{SUFFIX}_history.json'}
    return os.path.join(RESULTS, base[kind])


def stream(cmd):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in proc.stdout:
        print(line, end='', flush=True)
    return proc.wait()


for seed in SEEDS:
    tag = f'davis_random_binary_seed{seed}'
    if os.path.exists(final_name(tag, 'results')):
        print(f'seed {seed}: already done -> {os.path.basename(final_name(tag, "results"))}')
        continue

    shutil.rmtree(TMP, ignore_errors=True)
    os.makedirs(TMP)
    print(f'\n{"=" * 70}\nvolume-matched control, seed {seed}: random at {N_SUB:,} rows\n{"=" * 70}')
    code = stream([sys.executable, '-u', '-m', 'src.model.train',
                   '--split-dir', 'data/splits/davis/random', '--dataset', 'davis',
                   '--split', 'random', '--task', 'binary', '--seed', str(seed),
                   '--epochs', '100', '--min-epochs', '10', '--batch-size', str(BATCH),
                   '--train-subsample', str(N_SUB), '--results-dir', TMP])
    if code != 0:
        print(f'!! seed {seed} exited {code} -- not saved; re-run to retry')
        continue

    # Rename on the way to Drive, so these can never pass for the full-random cells.
    payload = json.load(open(os.path.join(TMP, f'{tag}_results.json')))
    assert payload['train_subsample'] == N_SUB and payload['n_train_rows'] == N_SUB, payload
    shutil.move(os.path.join(TMP, f'coldsite_dti_{tag}.pt'), final_name(tag, 'checkpoint'))
    shutil.move(os.path.join(TMP, f'coldsite_dti_{tag}_history.json'), final_name(tag, 'history'))
    payload['checkpoint'] = final_name(tag, 'checkpoint')
    payload['volume_matched_control'] = True
    with open(final_name(tag, 'results'), 'w') as f:
        json.dump(payload, f, indent=2)
    print(f'seed {seed}: saved -> {os.path.basename(final_name(tag, "results"))}  '
          f'(test AUROC {payload["test_metrics"]["auroc"]:.4f})')


## 6. Result

Compare these against ColdSite-DTI's full `random` and `cold_pair` cells from the Kaggle
grid once those exist. Three seeds is the minimum to quote a spread; a gap smaller than
that spread is not a finding.


In [ ]:
import statistics as st

aucs, n_rows = [], set()
for seed in SEEDS:
    path = final_name(f'davis_random_binary_seed{seed}', 'results')
    if not os.path.exists(path):
        print(f'seed {seed}: not done yet')
        continue
    payload = json.load(open(path))
    n_rows.add(payload['n_train_rows'])
    aucs.append(payload['test_metrics']['auroc'])
    print(f'seed {seed}: test AUROC {aucs[-1]:.4f}   best epoch {payload["best_epoch"]}   '
          f'trained on {payload["n_train_rows"]:,} rows')

assert n_rows <= {N_SUB}, f'a result trained on the wrong number of rows: {n_rows}'
if len(aucs) >= 2:
    print(f'\nvolume-matched random: AUROC {st.mean(aucs):.4f} +- {st.stdev(aucs):.4f} '
          f'over {len(aucs)} seeds, trained on {N_SUB:,} rows')
if len(aucs) == 3:
    print('\nAll three done. Send these numbers; they are compared against the grid\'s '
          'full-random and cold_pair ColdSite-DTI cells.')
